# 4. Procyclical Margins and Systemic Liquidity Externality

The final notebook explores volatility-linked IM and the assignment's systemic haircut extension. The systemic haircut responds to the previous period's fraction of investors selling assets.

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.otc_margining import ModelConfig,simulate_prices,simulate_default_days,simulate_liquidity,summarize_simulation
cfg=ModelConfig(paths=20000,horizon=252,maturity=300,mpor=10,seed=42)
default_days=simulate_default_days(cfg,seed=200)
reg_cfg=replace(cfg,annual_vol=.15)
reg_prices,vol_states=simulate_prices(reg_cfg,seed=300,extra_days=cfg.mpor,regime_switch=True,low_vol=.15,high_vol=.30)
fixed=simulate_liquidity(reg_prices,default_days,reg_cfg,mode='fire_sale')
dynamic=simulate_liquidity(reg_prices,default_days,reg_cfg,mode='fire_sale',volatility_states=vol_states)
pd.DataFrame([summarize_simulation(fixed,reg_cfg),summarize_simulation(dynamic,reg_cfg)],index=['Fixed 15% IM','Dynamic 15%/30% IM']).T

## Systemic haircut feedback

In [ ]:
prices=simulate_prices(cfg,seed=100,extra_days=cfg.mpor)
systemic=simulate_liquidity(prices,default_days,cfg,mode='fire_sale',systemic_h0=.10,systemic_theta=2.0)
print(summarize_simulation(systemic,cfg))
fig,ax=plt.subplots(figsize=(9,4.5))
ax.plot(systemic['systemic_haircut_path']*100,lw=1.5)
ax.set(xlabel='Trading day',ylabel='Fire-sale discount (%)',title='Lagged systemic fire-sale haircut')
plt.show()

## Interpretation and limitations

The haircut is capped at 50%. The model links fire-sale discounts through aggregate selling, but does not model correlated defaults or common asset-price returns. Results depend on selected parameters and simulation seeds.